### Imports and set identity

A custom environment was created with "edgartools" installed. This environment is used in this notebook and allows for an easy approach to extract inline XBRL tags from SEC DEF14A filings. The SEC requires a valid User‑Agent identity for automated access, which is set also.

In [1]:
# Imports and SEC identity setup
import requests
import pandas as pd
from datetime import datetime
from pyspark.sql import Row
from edgar import Company, set_identity

set_identity("stefan.werner@example.com")

StatementMeta(, fed34e0f-576d-4854-8e1b-b7ad755b8642, 5, Finished, Available, Finished, False)

### Initialize tables

Three Delta tables are created, if they don't exist. Delta gives ACID, schema evolution, time travel, and is the natural storage format in Fabric/Lakehouse. Schema is defined explicitely.

In [2]:
# Create reference tables if they do not exist
spark.sql("""
CREATE TABLE IF NOT EXISTS sec_all_tickers (
    ticker STRING,
    cik STRING,
    company_name STRING,
    first_seen_ts TIMESTAMP,
    last_seen_ts TIMESTAMP
)
USING delta
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_sec_company_def14a (
    ticker STRING,
    cik STRING,
    company_name STRING
)
USING delta
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_sec_company_no_def14a (
    ticker STRING,
    cik STRING,
    first_checked_ts TIMESTAMP,
    last_checked_ts TIMESTAMP
)
USING delta
""")


StatementMeta(, fed34e0f-576d-4854-8e1b-b7ad755b8642, 6, Finished, Available, Finished, False)

DataFrame[]

### Load SEC ticker universe and merge into sec_all_tickers

I want to extract as much executive / CEO compensation data as possible. To be able to do this, the official SEC ticker universe is pulled from company_tickers.json. This allows me to get a list for which companies exist, their CIKs and official names. SEC tickers are normalized to a 10-digit zero-padded string, as this is the consistent key format across the plattform. Then, first_seen_ts and last_seen_ts are captured for change tracking - when did ticker first appeared in the SEC universe, and when did they have been included the last time. A spark DataFrame is created with this information and exposed in a temp view for SQL. A Merge Upsert logic was used to update last_seen_ts for tickers that already exists and add a new row for new tickers.

In [3]:
# Load SEC ticker universe and MERGE into sec_all_tickers

url = "https://www.sec.gov/files/company_tickers.json"
headers = {"User-Agent": "stefan.werner@example.com"}

resp = requests.get(url, headers=headers)
data = resp.json()

now = datetime.utcnow()

rows = []
for entry in data.values():
    ticker = entry.get("ticker")
    if not ticker:
        continue
    cik = str(entry["cik_str"]).zfill(10)
    name = entry["title"]
    rows.append(Row(
        ticker=ticker,
        cik=cik,
        company_name=name,
        first_seen_ts=now,
        last_seen_ts=now
    ))

df_new = spark.createDataFrame(rows)
df_new.createOrReplaceTempView("sec_all_tickers_new")

spark.sql("""
MERGE INTO sec_all_tickers AS target
USING sec_all_tickers_new AS source
ON target.ticker = source.ticker
WHEN MATCHED THEN
  UPDATE SET target.last_seen_ts = source.last_seen_ts
WHEN NOT MATCHED THEN
  INSERT (ticker, cik, company_name, first_seen_ts, last_seen_ts)
  VALUES (source.ticker, source.cik, source.company_name,
          source.first_seen_ts, source.last_seen_ts)
""")


StatementMeta(, fed34e0f-576d-4854-8e1b-b7ad755b8642, 7, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Classify tickers

This step of the Bronze Layer classifies the complete SEC ticker universe into three mutually exclusive categories:
- Known DEF14A Companies: Companies for which at least one DEF 14A filing has already been identified.
- Known Non‑DEF14A Companies: Companies that have previously been checked and confirmed to have no DEF 14A filings.
- New Tickers: Companies that have never been evaluated and therefore require an initial DEF14A check.

This classification enables an incremental ingestion strategy. Only new tickers must be queried against the SEC API, significantly reducing external requests and improving pipeline performance.

In [4]:
# Classify tickers into: new, known DEF14A, known no-DEF14A

df_def = spark.sql("SELECT ticker FROM dim_sec_company_def14a")
def_tickers = {r["ticker"] for r in df_def.collect()}

df_no = spark.sql("SELECT ticker FROM dim_sec_company_no_def14a")
no_tickers = {r["ticker"] for r in df_no.collect()}

df_all = spark.sql("SELECT ticker, cik, company_name FROM sec_all_tickers")
all_rows = df_all.collect()

new_tickers = []
existing_def_tickers = []

for r in all_rows:
    t = r["ticker"]
    if t in def_tickers:
        existing_def_tickers.append((t, r["cik"], r["company_name"]))
    elif t in no_tickers:
        continue
    else:
        new_tickers.append((t, r["cik"], r["company_name"]))


StatementMeta(, fed34e0f-576d-4854-8e1b-b7ad755b8642, 8, Finished, Available, Finished, False)

### Checking new tickers for DEF14A filings

This step evaluates all previously unseen tickers (identified in the prior classification step) and determines whether they have DEF 14A filings.
The result updates two dimension tables. This ensures that each ticker is checked exactly once, enabling an efficient incremental ingestion pipeline.
Both dimension tables use a MERGE operation instead of simple APPEND writes.

In [5]:
# Check new tickers for DEF14A filings

from datetime import datetime

def check_def14a(ticker):
    """Return proxy object or None."""
    try:
        c = Company(ticker)
        filings = c.get_filings(form="DEF 14A")
        if not filings or filings.latest() is None:
            return None
        return filings.latest().obj()
    except:
        return None

now = datetime.utcnow()
new_def_rows = []
new_no_rows = []

for ticker, cik, name in new_tickers:
    proxy = check_def14a(ticker)

    if proxy is None:
        # No DEF14A found
        new_no_rows.append(Row(
            ticker=ticker,
            cik=cik,
            first_checked_ts=now,
            last_checked_ts=now
        ))
        continue

    # DEF14A exists → add to dimension
    new_def_rows.append(Row(
        ticker=ticker,
        cik=proxy.cik,
        company_name=proxy.company_name
    ))

# Update dim_sec_company_no_def14a
if new_no_rows:
    df_no_new = spark.createDataFrame(new_no_rows)
    df_no_new.createOrReplaceTempView("dim_sec_company_no_def14a_new")

    spark.sql("""
    MERGE INTO dim_sec_company_no_def14a AS target
    USING dim_sec_company_no_def14a_new AS source
    ON target.ticker = source.ticker
    WHEN NOT MATCHED THEN
    INSERT (ticker, cik, first_checked_ts, last_checked_ts)
    VALUES (source.ticker, source.cik, source.first_checked_ts, source.last_checked_ts)
    """)


if new_def_rows:
    df_def_new = spark.createDataFrame(new_def_rows)
    df_def_new.createOrReplaceTempView("dim_sec_company_def14a_new")

    spark.sql("""
    MERGE INTO dim_sec_company_def14a AS target
    USING dim_sec_company_def14a_new AS source
    ON target.ticker = source.ticker
    WHEN MATCHED THEN
    UPDATE SET 
        target.cik = source.cik,
        target.company_name = source.company_name
    WHEN NOT MATCHED THEN
    INSERT (ticker, cik, company_name)
    VALUES (source.ticker, source.cik, source.company_name)
    """)


StatementMeta(, fed34e0f-576d-4854-8e1b-b7ad755b8642, 9, Finished, Available, Finished, False)